# Chapter 1: Missing Values

###  The 3 Types of Missingness — This Is Exam Material

> This is one of the most conceptually important things in all of data science.
> **Why data is missing determines how you should handle it.**

| |  MCAR |  MAR |  MNAR |
|---|---|---|---|
| **Full Name** | Missing Completely At Random | Missing At Random | Missing Not At Random |
| **What it means** | Coin-flip missing | Depends on other columns | Missing because of its own value |
| **Example** | Sensor glitch | Income missing for young people | Sick people skip BMI |
| **How to handle** | Safe to drop rows | Impute using other cols | Needs domain knowledge |
| **Danger level** |  Safest type |  Common in surveys |  Creates bias |

>  **Danger increases left to right. MNAR can silently bias your model.**

###  MCAR — Missing Completely At Random
The missingness has **nothing to do with the data itself**.

> **Example:** A temperature sensor randomly drops a reading due to a power flicker.
> No pattern, no bias.

 You can safely **drop these rows** or **impute**.

---

###  MAR — Missing At Random
Missingness depends on **other observed columns**, not on the missing value itself.

> **Example:** In a survey, men are less likely to answer "weight" questions.
> The missingness depends on **gender** (visible) — not on the weight value itself.

 You can **impute using other columns**.

---

###  MNAR — Missing Not At Random
Missingness depends on **the value that's actually missing**.

> **Example:** Very sick patients skip their BMI entry because they're too ill.
> The missing BMI is **systematically lower** than observed BMIs —
> so dropping or simple imputation **introduces bias**.

 This is the **dangerous one**.
Requires **domain knowledge** and careful handling.

### Detecting Missing Values — Code

In [3]:
import pandas as pd
import numpy as np

# Sample dataset with missing values
data = {
    'Name':   ['Arjun', 'Priya', 'Rahul', None,    'Sneha'],
    'Age':    [25,       None,    32,      28,       None  ],
    'Salary': [50000,    62000,   None,    None,     45000 ],
    'City':   ['Hyd',   'Mum',   'Delhi', 'Chennai','Hyd' ]
}
df = pd.DataFrame(data)

In [4]:
# 1. total missings per column
print(df.isnull().sum())

Name      1
Age       2
Salary    2
City      0
dtype: int64


In [5]:
# 2 .Missing perecentage per columns
miss_pct = (df.isnull().sum()/len(df))*100
print(miss_pct)

Name      20.0
Age       40.0
Salary    40.0
City       0.0
dtype: float64


In [6]:
# 3. whicch rows have missing values
print(df[df.isnull().any(axis = 1)])

    Name   Age   Salary     City
1  Priya   NaN  62000.0      Mum
2  Rahul  32.0      NaN    Delhi
3   None  28.0      NaN  Chennai
4  Sneha   NaN  45000.0      Hyd


In [7]:
# 4 . heatmap style summary 
print(df.isnull())

    Name    Age  Salary   City
0  False  False   False  False
1  False   True   False  False
2  False  False    True  False
3   True  False    True  False
4  False   True   False  False


In [9]:
# 5 . total missing count across dataset
print(f"total missing count across dataset : {df.isnull().sum().sum()}")

total missing count across dataset : 5


In [10]:
# 6 . rows with no missing values
print(df.dropna())

    Name   Age   Salary City
0  Arjun  25.0  50000.0  Hyd


### Handling Missing values -- All strategies
### Strategy 1: Drop rows or columns

In [ ]:
# drop rows where any column is null 
dropped_df = df.dropna()

# drop rows where specific column is null
dropped_df = df.dropna(subset=['Age'])

# drop all columns which are null
dropped_df = df.dropna(how = 'all')

# drop columns with missing more than 40%
threshold = 40
dropped_df = df.loc[: , df.isnull().sum() < threshold]

# Rule of thumb
# - if missing in column is > 40% (Drop)
# - if target feature is null inthat row (Drop)

### Strategy 2 : Fill with fixed values

In [ ]:
# fill all nulls with a fixed value
df['Age'].fillna(0 , inplace = True)  # bad idea for age but valid for counts

# fill with string for categoricaal
df.['City'].fillna('Unknown' , inplace = True)

# fill forward ( previous rows value ) - good for timeseries
df.['Salary'].fillna(method = 'ffill' , inplace = True)

# fill backward (next rows value)
df['Salary'].fillns(method = 'bfill' , inplace = True)

### Strategy 3 : Statistical Imputation

In [ ]:
# fill with mean  - use for normally distributed numerical columns 
mean_age = df['Age'].mean()
df['Age'].fillna(mean_age , inplace = True)

# fill with median - best for numerical columns which has outliners(skewed)
median_age = df['Salary'].median()
df['Salary'].fillna(median_age , inplace = True)

# fill with mode - use for categoriacl columns
mode_city = df['City'].mode()
df['City'].fillna(mode_city , inplace = True)

# when to use MEAN VS MEDIAN
# - salary data : use median (outliner like salary of ceo skews mean)
# - height data : use mean ( normally distributed , no outliners)

### Strategy 4 : Scikit-learn SimpleImputer (production grade)

In [ ]:
from sklearn.impute import SimpleImputer

# for numerical columns 
num_imputer = SimpleImputer(strategy = 'median')  # or 'mean' or 'most_frequent'
df[['Age' , 'Salary']] = num_imputer.fit_transform(df[['Age' , 'Salary']])

# for categorical columns 
cat_imputer = SimpleImputer(strategy = 'most_frequent')
df[['City' , 'Name']] = cat_imputer.fit_transform(df[['City' , 'Name']])

# why SSimpleImputer instead of fillna ? 
#  beacause in a pipeline fit() learns from a training data and
# transform () applies the same values to test data - preventing data leakage

### Strategy 5 : KNN imputation

In [ ]:
from sklearn.impute import KNNImputer
# KNN finds the K-nearest rows and fills missing values
# using the average of those neighbours 

knn_imnputer = KNNImputer(n_neighbors = 5)
df_numeric = df[['Age' , 'Salary']]
df_imputed = knn_imnputer.fit_transform(df_numeric)

# EX : if a row has missing age but have salary = '80000'
# KNN finds 5 rows with similar salaries and average their ages 
# much smarter than Mean / Median 

# Chapter 2 : Outliner Detection 
An outlier is a data point that is far away from the rest. But is it an error (typo: age = 999) or
a genuine extreme (CEO salary = ₹5 crore)? You must decide based on domain knowledge.

### Method 1 : The IQR (interquartile range) - The standatd method

In [15]:
import numpy as np
import pandas as pd

# Load titanic for example
df = pd.read_csv('../data/titanic.csv')

def find_outliners_iqr(df , column):
    Q1 = df[column].quantile(0.25)  # 25th percentile
    Q3 = df[column].quantile(0.75)  # 75th percentile

    IQR = Q3 - Q1

    lower_fence = Q1 - 1.5 * IQR
    upper_fence = Q3 + 1.5 * IQR

    outliners = df[(df[column] < lower_fence) | (df[column] > upper_fence)]

    print(f"column : {column}" )
    print(f"Q1 = {Q1:.2f} , Q3 = {Q3:.2f} , IQR = {IQR:.2f}")
    print(f"Lower Fence = {lower_fence:.2f}")
    print(f"Upper Fence = {upper_fence:.2f}")
    print(f"Outliners Found = {len(outliners)}")
    return outliners

fare_outliners = find_outliners_iqr(df , 'Fare')
age_outliners = find_outliners_iqr(df , 'Age')

#The IQR method is like a "fence". 
#Any value beyond 1.5 × IQR from Q1 or Q3 is flagged. Use 3.0 × IQR for a stricter fence (only extreme outliers).

column : Fare
Q1 = 7.91 , Q3 = 31.00 , IQR = 23.09
Lower Fence = -26.72
Upper Fence = 65.63
Outliners Found = 116
column : Age
Q1 = 20.12 , Q3 = 38.00 , IQR = 17.88
Lower Fence = -6.69
Upper Fence = 64.81
Outliners Found = 11


### Method 2 : Z-Score - For Normally Distributed Data

In [17]:
from scipy import stats 

def find_outliners_zscore(df , column , threshold = 3):
    z_score = np.abs(stats.zscore(df[column].dropna()))
    # z - score = how many standard devbiations from mean 
    # |z| > 3 means the value is 3 standard deeviations away - very unusual
    outliner_indices = np.where(z_score > threshold)[0]
    outliners = df[column].dropna().iloc[outliner_indices]

    print(f"column : {column}")
    print(f"mean : {df[column].mean():.2f}, std : {df[column].std():.2f}")
    print(f"outliners (|z| > {threshold}): {len(outliners)}")
    return outliners

age_outliners = find_outliners_zscore(df , 'Age')
fare_outliners = find_outliners_zscore(df , 'Fare')

# when to use z-score vs iqr
# - z score : Assumes normal disstribution.good for age , height 
# - iqr : no distribution assumption . better for skewed data 

column : Age
mean : 29.70, std : 14.53
outliners (|z| > 3): 2
column : Fare
mean : 32.20, std : 49.69
outliners (|z| > 3): 20


### Method 3 : Isolation Forest — For Complex , High-Dimensional Data

In [21]:
from sklearn.ensemble import IsolationForest

# isolation forest is a machine learning algorithm for outliners
# idea =  outliners are easy to "isolate" in a decision tree 
# they end up in a short path because they are far from clusters 

iso_forest = IsolationForest(
    contamination = 0.05,  # we assume data have 5 % of outliners
    random_state = 42
)

# works on multiple columns simultaneously
features = df[['Age','Fare']].dropna()
predictions = iso_forest.fit_predict(features)

# returns 1 = normal , -1 = outliner

outliner_mask = predictions == -1
print(f"isolation forest found {outliner_mask.sum()} outliners")
print(features[outliner_mask].head(10))

# use when if :
# u have many features
# u suspect multidimensinal outliners

isolation forest found 36 outliners
      Age      Fare
27   19.0  263.0000
54   65.0   61.9792
88   23.0  263.0000
96   71.0   34.6542
116  70.5    7.7500
118  24.0  247.5208
195  58.0  146.5208
258  35.0  512.3292
268  58.0  153.4625
297   2.0  151.5500


### Handling Outliers — What To Do After Detection

In [ ]:

# option 1 ; replpace outliner with fence 
def cap_outliners(df , column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1
    lower =  Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[column] = df[column].clip(lower = lower , upper = upper)
    return df


# option 2 : drop outilners rows only if MCAR - random errors
df_clean = df[df['Fare'] < 300]  # drop exteme fares

# option 3 : log transformation - reduces imapct of outliners 
df['Fare_log'] = np.log1p(df['Fare'])

# option 4 : keep them  - if outliners are genuine 
# document them - use robust mdoels ( tree based , not linear regression)
# contextual error - domain expert decision


#  Chapter 3: Duplicate Detection and Handling

In [ ]:

# 1.check total duplicates
print(f"duplicate rows : {df.duplicated().sum()}")

# 2.see the sctual dupliacted rows 
print(df[df.duplicated(keep = False)])  # - keep = false shows all copies

# 3. duplicated bases on specific columnns 
# maybe passengerId is unique but name + age combo is duplicated 
print(df.duplicated(subset = ['Name' , 'Age']).sum())

# 4. drop dupicates
df = df.drop_duplicates()
df = df.drop_duplicates(keep = 'last')  # keep last occurances
df = drop_duplicates(keep = False)      # drop all copies even originals

# 5. duplicates on subsets of columns 
df = df.drop_duplicates(subset = ['PassengerId'] , keep = 'first')

# 6. reset index after dropping duplicates 
df = df.drop_duplicates().reset_index(drop = True) # without reset index skip - confusing

# Chapter 4 : Data Types corrections

In [ ]:
# 1. detect wrong datatypes
print(df.dtypes)

# sometimes 'Age' , 'Salaries' may look like object

# 2 . why does this happens
# someone enterd "N/A" or "unknown" in numeric
# pandas reads the whole columns as object(string) 

#check unique values to spot out culprits
print(df['Age'].unique())

# 3. fix: replace non-numeric strings , then convert
df['Age'] = df['Age'].replace({'N/A' : np.nan , 'unknown' : np.nan , '--' : np.nan})
df['Age'] = pd.to_numeric(df['Age'] , errors = 'coerce')

# errors = 'coerce' ->any value that can't be converted becomes NaN

# 4. fix salary stores as string with commas 
df['Salary'] = df['Salary'].str.replace(',', '').astype(float)

# 5 . fix boolean stored as yes/no strings 
df['Survived'] = df['Surivived'].map({'yes' : True , 'no ' : False})
df['Churn'] = df['Churn'].replace({'Yes' : 1 , 'No' : 0}).astype(int)

# 6. fix  date and time stored as string 
df['JoinDate'] = pd.to_datetime(df['JoinDate'], format='%d-%m-%Y') # '%d-%m-%Y' means: 25-08-2023

# after converting u can extract  parts :
df['Year'] = df['JoinDate'].dt.year
df['Month'] = df['JoinDate'].dt.month
df['Day'] = df['JoinDate'].dt.day


# 7 . convert to category dtype (saves memory)
df['Sex']      = df['Sex'].astype('category')
df['Embarked'] = df['Embarked'].astype('category')


# Chapter 5 : Inconcisitent Values

In [ ]:
# Inconsistency means the same thing written in different ways. "Mumbai", "mumbai", "MUMBAI"
# — all mean the same city, but pandas treats them as 3 different categories.

# 1. detect inconsistents
print(df['City'].value_counts())

# 2. fix inconsistencies 
df['City'] = df['City'].str.strip()
df['City'] = df['City'].str.title()

# or 

df['City'] = df['City'].str.lower()
df['City'] = df['City'].str.upper()

# 3 . fiic=x abbrevations using replace 
city_map = {
    'Hyd' : 'Hyderabad',
    'hyderabad' : 'Hydereabad',
    'mumbai' : 'Mumbai'
    'Mum' : 'Mumbai'
}
df['City'] = df.replace(city_map)

# 4, fix extra white space hidden inside string 
df['Name'] = df['Name'].str.strip()
df['Name'] = df['Name'].str.replace(r'\s+' , ' ' , regex = True) # colapse multiple spaces


# 5 . fix impossible values 
# age cant be negative or > 120 
df.loc[df['Age'] < 0 , 'Age'] = np.nan
df.loc[df['Age'] > 120 , 'Age'] = np.nan

# salary cant be negative
df.loc[df['Salary'] < 0, 'Salary'] = np.nan

# 6. check for invalid email format 
import re
def is_valid_email(email):
    if pd.isna(email):
        return False
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern , str(email)))

df['email_valid'] = df['Email'].apply(is_valid_email)
print(df[~df['email_valid']]) 


# END OF  noteboook 2 